In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Unzip from Drive straight to Colab's high-speed local storage
!unzip -q "/content/drive/MyDrive/Colab Notebooks/covid detection/archive.zip" -d "/content/dataset/"

Mounted at /content/drive


In [2]:
import os
import tensorflow as tf
import keras
from keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from keras.applications import Xception
from keras.applications.xception import preprocess_input
from keras.models import Model
from keras.optimizers import Adam
from keras.preprocessing import image_dataset_from_directory

In [3]:
# Define paths and constants
TRAIN_DIR = 'dataset/Data/train/'
TEST_DIR = 'dataset/Data/test/'
IMG_SIZE = (224, 224)  # Xception expected input size
BATCH_SIZE = 32
NUM_CLASSES = 3        # COVID19, NORMAL, PNEUMONIA

In [4]:
# Setup ImageDataGenerators with Xception preprocessing
# Data Augmentation for training to prevent overfitting
train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='training',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    label_mode='categorical'  # For multi-class classification
)

# Only preprocessing (no augmentation) for testing/validation
val_ds = image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='validation',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    label_mode='categorical'  # For multi-class classification
)

test_ds = image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode='categorical'  # For multi-class classification
)

train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

Found 5144 files belonging to 3 classes.
Using 4116 files for training.
Found 5144 files belonging to 3 classes.
Using 1028 files for validation.
Found 1288 files belonging to 3 classes.


In [5]:
# Build Xception Transfer Learning Model
input_shape = IMG_SIZE + (3,)
base_model = Xception(weights='imagenet', include_top=False, input_shape=input_shape)

# Freeze the base Xception layers
base_model.trainable = False

# Build custom network head
inputs = Input(shape=input_shape)
x = preprocess_input(inputs)
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)

model.summary()

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ xception (Functional)           │ (None, 7, 7, 2048)     │    20,861,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,386,795 (81.58 MB)

 Trainable params: 525,315 (2.00 MB)

 Non-trainable params: 20,861,480 (79.58 MB)

In [6]:
# Compile Model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [7]:
# Train the Model
EPOCHS = 10
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)

Epoch 1/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 76s 455ms/step - accuracy: 0.7816 - loss: 3.0001 - val_accuracy: 0.9076 - val_loss: 0.2909
Epoch 2/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 29s 150ms/step - accuracy: 0.8423 - loss: 0.4254 - val_accuracy: 0.9280 - val_loss: 0.2311
Epoch 3/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 20s 154ms/step - accuracy: 0.8545 - loss: 0.3989 - val_accuracy: 0.9154 - val_loss: 0.2258
Epoch 4/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 20s 158ms/step - accuracy: 0.8727 - loss: 0.3554 - val_accuracy: 0.9222 - val_loss: 0.2119
Epoch 5/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 21s 161ms/step - accuracy: 0.8700 - loss: 0.3681 - val_accuracy: 0.9261 - val_loss: 0.2033
Epoch 6/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 21s 165ms/step - accuracy: 0.8848 - loss: 0.3315 - val_accuracy: 0.9377 - val_loss: 0.1740
Epoch 7/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 22s 173ms/step - accuracy: 0.8732 - loss: 0.3482 - val_accuracy: 0.9348 - val_loss: 0.1913
Epoch 8/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 25s 191ms/step - accuracy: 0.8822 - loss: 0

In [8]:
import numpy as np
import seaborn as sns
import sklearn.metrics as metrics
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Overall Performance Evaluation
loss, accuracy = model.evaluate(test_ds)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

41/41 ━━━━━━━━━━━━━━━━━━━━ 17s 362ms/step - accuracy: 0.9177 - loss: 0.2302

Test Loss: 0.2302
Test Accuracy: 91.77%


In [9]:
# Save the trained model
model.save('covid_pneumonia_xception_model.keras')
print("Model saved successfully!")

Model saved successfully!
